In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from preprocessing import filter_all_channels
from features import extract_all_windows
from config import N_SUBJECTS, N_CLASSES, get_subject_path

def run_subject_pipeline(subject_id):
    """
    Run full preprocessing and classification pipeline for one subject.
    
    Args:
        subject_id: integer 1-40
    
    Returns:
        mean accuracy, std accuracy, or None if file not found
    """
    path = get_subject_path(subject_id, exercise=1)
    
    try:
        data = sio.loadmat(path)
    except FileNotFoundError:
        print(f"S{subject_id}: file not found, skipping")
        return None, None
    
    emg = data['emg']
    labels = data['restimulus']
    
    # Full pipeline
    emg_filtered = filter_all_channels(emg)
    X, y = extract_all_windows(emg_filtered, labels)
    
    svm_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel='rbf', C=1.0, gamma='scale'))
    ])
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(svm_pipeline, X, y, cv=cv, scoring='accuracy')
    
    return scores.mean(), scores.std()

print("Starting multi-subject evaluation...")
print(f"{'Subject':<10} {'Accuracy':<12} {'Std':<8}")
print("-" * 30)

In [ ]:
results = {}

for subject_id in range(1, N_SUBJECTS + 1):
    mean_acc, std_acc = run_subject_pipeline(subject_id)
    
    if mean_acc is not None:
        results[subject_id] = (mean_acc, std_acc)
        print(f"S{subject_id:<9} {mean_acc:.3f}        ±{std_acc:.3f}")
    
print("-" * 30)

# Summary statistics
accuracies = [v[0] for v in results.values()]
print(f"\nCompleted: {len(results)}/40 subjects")
print(f"Mean accuracy: {np.mean(accuracies):.3f} ± {np.std(accuracies):.3f}")
print(f"Best subject:  S{max(results, key=lambda k: results[k][0])} — {max(accuracies):.3f}")
print(f"Worst subject: S{min(results, key=lambda k: results[k][0])} — {min(accuracies):.3f}")

In [ ]:
subjects = list(results.keys())
accuracies = [results[s][0] for s in subjects]
stds = [results[s][1] for s in subjects]

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Bar chart per subject
axes[0].bar(subjects, accuracies, yerr=stds, capsize=3, 
            color='steelblue', alpha=0.8)
axes[0].axhline(np.mean(accuracies), color='red', 
                linestyle='--', label=f'Mean: {np.mean(accuracies):.3f}')
axes[0].set_xlabel('Subject')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('SVM Classification Accuracy per Subject (DB2, Exercise 1)')
axes[0].legend()
axes[0].set_ylim(0, 1)

# Distribution
axes[1].hist(accuracies, bins=10, color='steelblue', 
             alpha=0.8, edgecolor='black')
axes[1].axvline(np.mean(accuracies), color='red', 
                linestyle='--', label=f'Mean: {np.mean(accuracies):.3f}')
axes[1].set_xlabel('Accuracy')
axes[1].set_ylabel('Number of subjects')
axes[1].set_title('Distribution of Accuracy Across Subjects')
axes[1].legend()

plt.tight_layout()
plt.show()